# Django, Views

## Introduction to Django Views
---

In this lesson, you'll learn how to write **Django views** - the heart of your application's logic.

Views are Python functions or classes that receive HTTP requests and return HTTP responses.

1. [Function-Based Views (FBV)](#Function-Based-Views-(FBV)),
    - [Basic Structure](#Basic-Structure),
    - [Returning Different Response Types](#Returning-Different-Response-Types),
2. [Request and Response Objects](#Request-and-Response-Objects),
    - [The Request Object](#The-Request-Object),
    - [The Response Object](#The-Response-Object),
3. [Class-Based Views (CBV)](#Class-Based-Views-(CBV)),
    - [Basic CBV Structure](#Basic-CBV-Structure),
    - [HTTP Method Handlers](#HTTP-Method-Handlers),
4. [FBV vs CBV - When to Use Which](#FBV-vs-CBV---When-to-Use-Which),
    - [Comparison](#Comparison),
    - [Practical Guidelines](#Practical-Guidelines),
5. [🧠 EXERCISE 🧠, Create Views for Bookstore](#🧠-EXERCISE-🧠,-Create-Views-for-Bookstore).

<br>

## Function-Based Views (FBV)

---

### Basic Structure

---

A **function-based view** (FBV) is simply a Python function that:

1. Takes an `HttpRequest` object as the first argument
2. Returns an `HttpResponse` object

<br>

```
Request  →  View Function  →  Response
```

In [ ]:
# Example: Simplest possible view

from django.http import HttpRequest, HttpResponse

def hello(request: HttpRequest) -> HttpResponse:
    """Return a simple greeting."""
    return HttpResponse("Hello, World!")

In [ ]:
# Example: View with URL parameter

from django.http import HttpRequest, HttpResponse

def greet_user(request: HttpRequest, name: str) -> HttpResponse:
    """Greet a specific user by name."""
    return HttpResponse(f"Hello, {name}!")

# URL pattern: path('greet/<str:name>/', views.greet_user)
# Visit: /greet/Alice/  →  "Hello, Alice!"

In [ ]:
# Example: View with multiple parameters

from django.http import HttpRequest, HttpResponse

def book_detail(request: HttpRequest, pk: int) -> HttpResponse:
    """Display book details by primary key."""
    # In real app, you'd fetch from database
    books = {
        1: "Django for Beginners",
        2: "Two Scoops of Django",
        3: "Django for Professionals",
    }
    
    title = books.get(pk, "Book not found")
    return HttpResponse(f"<h1>{title}</h1>")

<br>

### Returning Different Response Types

---

Views can return different types of responses:

In [ ]:
# Example: Different response types

from django.http import HttpResponse, JsonResponse, HttpResponseRedirect, Http404
from django.shortcuts import render, redirect, get_object_or_404

# 1. Plain text
def plain_text(request):
    return HttpResponse("Just plain text", content_type="text/plain")

# 2. HTML
def html_response(request):
    return HttpResponse("<h1>HTML Heading</h1>")

# 3. JSON (for APIs)
def json_response(request):
    data = {'name': 'Django', 'version': '5.0'}
    return JsonResponse(data)

# 4. Redirect
def redirect_view(request):
    return redirect('book_list')  # Redirects to named URL

In [ ]:
# Example: Rendering templates (most common)

from django.shortcuts import render

def book_list(request):
    """Display list of books using a template."""
    books = [
        {'id': 1, 'title': 'Django for Beginners', 'price': 29.99},
        {'id': 2, 'title': 'Two Scoops of Django', 'price': 49.99},
        {'id': 3, 'title': 'Django for Professionals', 'price': 39.99},
    ]
    
    context = {
        'books': books,
        'title': 'Our Books',
    }
    
    return render(request, 'catalog/book_list.html', context)

In [ ]:
# Example: Handling errors

from django.http import Http404, HttpResponseNotFound, HttpResponseForbidden

def book_detail(request, pk: int):
    """Display book or return 404."""
    books = {1: 'Book One', 2: 'Book Two'}
    
    if pk not in books:
        # Option 1: Raise Http404 exception
        raise Http404(f"Book {pk} not found")
    
    return HttpResponse(f"<h1>{books[pk]}</h1>")

def protected_view(request):
    """Return 403 Forbidden for unauthorized access."""
    if not request.user.is_staff:
        return HttpResponseForbidden("Staff only!")
    return HttpResponse("Welcome, staff member!")

<br>

## Request and Response Objects

---

### The Request Object

---

The `HttpRequest` object contains all information about the incoming request:

<br>

| Attribute | Description | Example |
|-----------|-------------|---------|
| `request.method` | HTTP method | `'GET'`, `'POST'` |
| `request.path` | URL path | `'/books/5/'` |
| `request.GET` | Query parameters | `{'page': '2'}` |
| `request.POST` | POST data | `{'title': 'New Book'}` |
| `request.user` | Current user | `<User: alice>` |
| `request.session` | Session data | `{'cart': [...]}` |
| `request.COOKIES` | Cookies | `{'csrftoken': '...'}` |
| `request.META` | HTTP headers | `{'HTTP_HOST': '...'}` |

In [ ]:
# Example: Using request attributes

from django.http import HttpRequest, HttpResponse

def debug_request(request: HttpRequest) -> HttpResponse:
    """Show request information for debugging."""
    info = f"""
    Method: {request.method}
    Path: {request.path}
    Full URL: {request.build_absolute_uri()}
    User: {request.user}
    Is authenticated: {request.user.is_authenticated}
    """
    return HttpResponse(f"<pre>{info}</pre>")

In [ ]:
# Example: Working with query parameters

from django.http import HttpRequest, HttpResponse

def search_books(request: HttpRequest) -> HttpResponse:
    """Search books with query parameters."""
    # URL: /books/search/?q=django&page=2
    
    query = request.GET.get('q', '')          # 'django' or empty string
    page = request.GET.get('page', '1')       # '2' or '1' (default)
    sort = request.GET.get('sort', 'title')   # 'title' (default)
    
    # Convert page to int safely
    try:
        page = int(page)
    except ValueError:
        page = 1
    
    return HttpResponse(f"Searching for '{query}', page {page}, sort by {sort}")

In [ ]:
# Example: Handling POST data

from django.http import HttpRequest, HttpResponse
from django.views.decorators.http import require_http_methods

@require_http_methods(["GET", "POST"])
def contact_form(request: HttpRequest) -> HttpResponse:
    """Handle contact form submission."""
    
    if request.method == 'POST':
        # Process form data
        name = request.POST.get('name', '')
        email = request.POST.get('email', '')
        message = request.POST.get('message', '')
        
        # Validate and save (simplified)
        if name and email and message:
            return HttpResponse(f"Thanks, {name}! We'll contact you at {email}.")
        else:
            return HttpResponse("Please fill all fields.", status=400)
    
    # GET request - show the form
    return HttpResponse("""
        <form method="post">
            <input name="name" placeholder="Name"><br>
            <input name="email" placeholder="Email"><br>
            <textarea name="message"></textarea><br>
            <button type="submit">Send</button>
        </form>
    """)

<br>

### The Response Object

---

The `HttpResponse` object represents the response you send back:

<br>

| Method/Attribute | Description |
|------------------|-------------|
| `HttpResponse(content)` | Create response with content |
| `response.status_code` | HTTP status code (200, 404, etc.) |
| `response['Header']` | Set response headers |
| `response.set_cookie()` | Set a cookie |

In [ ]:
# Example: Customizing responses

from django.http import HttpResponse

def custom_response(request):
    """Return a customized response."""
    response = HttpResponse("Custom response")
    
    # Set status code
    response.status_code = 200
    
    # Set headers
    response['X-Custom-Header'] = 'Hello'
    response['Content-Type'] = 'text/plain'
    
    # Set cookie
    response.set_cookie('visited', 'true', max_age=3600)  # 1 hour
    
    return response

In [ ]:
# Example: Common response shortcuts

from django.http import (
    HttpResponse,           # 200 OK
    HttpResponseRedirect,   # 302 Found (redirect)
    HttpResponseNotFound,   # 404 Not Found
    HttpResponseForbidden,  # 403 Forbidden
    HttpResponseBadRequest, # 400 Bad Request
    JsonResponse,           # JSON with correct content-type
)

# Or use status parameter
def created_response(request):
    return HttpResponse("Resource created", status=201)

def no_content_response(request):
    return HttpResponse(status=204)

<br>

## Class-Based Views (CBV)

---

### Basic CBV Structure

---

A **class-based view** (CBV) is a Python class that handles HTTP requests:

- Inherits from `View` (or its subclasses)
- Defines methods for HTTP verbs: `get()`, `post()`, `put()`, `delete()`, etc.
- Uses `.as_view()` in URL configuration

In [ ]:
# Example: Basic class-based view

from django.http import HttpRequest, HttpResponse
from django.views import View

class HelloView(View):
    """A simple class-based view."""
    
    def get(self, request: HttpRequest) -> HttpResponse:
        """Handle GET requests."""
        return HttpResponse("Hello from CBV!")

# In urls.py:
# path('hello/', HelloView.as_view(), name='hello')

In [ ]:
# Example: CBV with URL parameter

from django.http import HttpRequest, HttpResponse
from django.views import View

class BookDetailView(View):
    """Display book details."""
    
    def get(self, request: HttpRequest, pk: int) -> HttpResponse:
        """Handle GET request for book detail."""
        # URL parameters are passed as keyword arguments
        return HttpResponse(f"Book with ID: {pk}")

# In urls.py:
# path('books/<int:pk>/', BookDetailView.as_view(), name='book_detail')

<br>

### HTTP Method Handlers

---

CBVs handle different HTTP methods with separate methods:

| HTTP Method | CBV Method |
|-------------|------------|
| GET | `get(self, request, *args, **kwargs)` |
| POST | `post(self, request, *args, **kwargs)` |
| PUT | `put(self, request, *args, **kwargs)` |
| PATCH | `patch(self, request, *args, **kwargs)` |
| DELETE | `delete(self, request, *args, **kwargs)` |

In [ ]:
# Example: Handling GET and POST in a CBV

from django.http import HttpRequest, HttpResponse
from django.views import View
from django.shortcuts import render

class ContactView(View):
    """Handle contact form with GET and POST."""
    
    template_name = 'contact.html'
    
    def get(self, request: HttpRequest) -> HttpResponse:
        """Display the contact form."""
        return render(request, self.template_name)
    
    def post(self, request: HttpRequest) -> HttpResponse:
        """Process the contact form submission."""
        name = request.POST.get('name')
        email = request.POST.get('email')
        message = request.POST.get('message')
        
        # Process the data (save to DB, send email, etc.)
        
        return HttpResponse(f"Thanks for your message, {name}!")

In [ ]:
# Example: CBV with shared setup logic

from django.http import HttpRequest, HttpResponse
from django.views import View

class BookView(View):
    """Handle CRUD operations for a book."""
    
    # Class attribute - shared data
    books = {
        1: {'title': 'Django Basics', 'price': 29.99},
        2: {'title': 'Advanced Django', 'price': 49.99},
    }
    
    def setup(self, request: HttpRequest, *args, **kwargs):
        """Called before dispatch - initialize shared state."""
        super().setup(request, *args, **kwargs)
        # Any setup logic here
    
    def get(self, request: HttpRequest, pk: int) -> HttpResponse:
        """Get book details."""
        book = self.books.get(pk)
        if not book:
            return HttpResponse("Not found", status=404)
        return HttpResponse(f"{book['title']} - ${book['price']}")
    
    def delete(self, request: HttpRequest, pk: int) -> HttpResponse:
        """Delete a book."""
        if pk in self.books:
            del self.books[pk]
            return HttpResponse(status=204)  # No content
        return HttpResponse("Not found", status=404)

In [ ]:
# Example: Using TemplateView for simple template rendering

from django.views.generic import TemplateView

class AboutView(TemplateView):
    """Simple template-based view."""
    template_name = 'about.html'
    
    def get_context_data(self, **kwargs):
        """Add extra context to the template."""
        context = super().get_context_data(**kwargs)
        context['company_name'] = 'Bookstore Inc.'
        context['founded'] = 2020
        return context

# Even simpler - directly in urls.py:
# path('about/', TemplateView.as_view(template_name='about.html'), name='about')

<br>

## FBV vs CBV - When to Use Which

---

### Comparison

---

| Aspect | Function-Based Views | Class-Based Views |
|--------|---------------------|-------------------|
| **Simplicity** | ✅ Easy to understand | ❌ More complex |
| **Explicit** | ✅ All logic visible | ❌ Hidden in inheritance |
| **Reusability** | ❌ Harder to reuse | ✅ Inheritance, mixins |
| **HTTP methods** | Manual if/else | Separate methods |
| **Decorators** | Simple to apply | Use `method_decorator` |
| **Testing** | Straightforward | May need more setup |

In [ ]:
# Comparison: Same functionality - FBV vs CBV

# === FBV approach ===
from django.http import HttpResponse
from django.views.decorators.http import require_http_methods

@require_http_methods(["GET", "POST"])
def book_form_fbv(request, pk=None):
    """Handle book form - create or edit."""
    if request.method == 'GET':
        # Show form
        return HttpResponse("Showing book form")
    else:  # POST
        # Process form
        return HttpResponse("Processing book form")

In [ ]:
# === CBV approach ===
from django.http import HttpResponse
from django.views import View

class BookFormView(View):
    """Handle book form - create or edit."""
    
    def get(self, request, pk=None):
        """Show the form."""
        return HttpResponse("Showing book form")
    
    def post(self, request, pk=None):
        """Process the form."""
        return HttpResponse("Processing book form")

<br>

### Practical Guidelines

---

**Use FBV when:**

- Simple, one-off views
- Learning Django (easier to understand)
- Views with complex, custom logic
- Single HTTP method handling

<br>

**Use CBV when:**

- CRUD operations (use generic views)
- Need to reuse code across views
- Multiple HTTP methods with shared setup
- Following RESTful patterns

In [ ]:
# Example: When FBV is cleaner

from django.http import JsonResponse

def health_check(request):
    """Simple health check endpoint."""
    return JsonResponse({'status': 'ok'})

# No need for a class here - it would be overkill

In [ ]:
# Example: When CBV shines - reusable base class

from django.views import View
from django.http import JsonResponse

class APIView(View):
    """Base class for API views."""
    
    def dispatch(self, request, *args, **kwargs):
        """Add common API logic."""
        # Check API key, rate limiting, etc.
        return super().dispatch(request, *args, **kwargs)
    
    def json_response(self, data, status=200):
        """Helper method for JSON responses."""
        return JsonResponse(data, status=status)


class BookAPIView(APIView):
    """Book API - inherits common API logic."""
    
    def get(self, request, pk=None):
        if pk:
            return self.json_response({'id': pk, 'title': 'Book'})
        return self.json_response({'books': []})


class AuthorAPIView(APIView):
    """Author API - also inherits common API logic."""
    
    def get(self, request, pk=None):
        return self.json_response({'authors': []})

<br>

## Useful View Decorators and Shortcuts

---

In [ ]:
# Common decorators for FBVs

from django.views.decorators.http import require_GET, require_POST, require_http_methods
from django.views.decorators.csrf import csrf_exempt
from django.contrib.auth.decorators import login_required

@require_GET
def list_view(request):
    """Only accepts GET requests."""
    return HttpResponse("List")

@require_POST
def create_view(request):
    """Only accepts POST requests."""
    return HttpResponse("Created")

@require_http_methods(["GET", "POST"])
def form_view(request):
    """Accepts GET and POST only."""
    return HttpResponse("Form")

@login_required
def protected_view(request):
    """Requires user to be logged in."""
    return HttpResponse(f"Hello, {request.user}!")

In [ ]:
# Common shortcuts

from django.shortcuts import render, redirect, get_object_or_404, get_list_or_404

def book_detail(request, pk):
    """Using get_object_or_404."""
    # Raises Http404 if not found
    # book = get_object_or_404(Book, pk=pk)
    # return render(request, 'book_detail.html', {'book': book})
    pass

def redirect_example(request):
    """Using redirect shortcut."""
    # Redirect to named URL
    return redirect('catalog:book_list')
    
    # Or to a URL path
    # return redirect('/books/')
    
    # Or to an object with get_absolute_url()
    # return redirect(book)

<br>

## Summary

---

| Concept | Key Points |
|---------|------------|
| **FBV** | Simple function: `def view(request)` → `HttpResponse` |
| **CBV** | Class with methods: `get()`, `post()`, etc. |
| **Request object** | `request.method`, `.GET`, `.POST`, `.user` |
| **Response object** | `HttpResponse`, `JsonResponse`, `redirect()` |
| **render()** | Shortcut to render template with context |
| **URL parameters** | Passed as function/method arguments |
| **When FBV** | Simple views, custom logic, learning |
| **When CBV** | CRUD, reusability, multiple HTTP methods |

<br>

### 🧠 EXERCISE 🧠, Create Views for Bookstore

---

Using the `bookstore` project:

1. Create a FBV `book_list` that returns a list of books as JSON
2. Create a FBV `book_detail` that takes `pk` and returns book info or 404
3. Create a CBV `BookSearchView` that:
   - GET: returns search form (just text for now)
   - POST: returns search results based on `request.POST.get('query')`
4. Add URL patterns for all views
5. Test all endpoints with the browser and/or curl

<details>
    <summary>▶️ Solution</summary>
    
**1-3. Edit `catalog/views.py`:**

```python
from django.http import HttpResponse, JsonResponse, Http404
from django.views import View

# Sample data
BOOKS = {
    1: {'id': 1, 'title': 'Django for Beginners', 'author': 'William Vincent'},
    2: {'id': 2, 'title': 'Two Scoops of Django', 'author': 'Daniel Feldroy'},
    3: {'id': 3, 'title': 'Django for Professionals', 'author': 'William Vincent'},
}

# 1. FBV - List all books
def book_list(request):
    return JsonResponse({'books': list(BOOKS.values())})

# 2. FBV - Book detail
def book_detail(request, pk: int):
    book = BOOKS.get(pk)
    if book is None:
        raise Http404(f"Book {pk} not found")
    return JsonResponse(book)

# 3. CBV - Book search
class BookSearchView(View):
    def get(self, request):
        return HttpResponse("""
            <form method="post">
                <input name="query" placeholder="Search books...">
                <button type="submit">Search</button>
            </form>
        """)
    
    def post(self, request):
        query = request.POST.get('query', '').lower()
        results = [
            book for book in BOOKS.values()
            if query in book['title'].lower() or query in book['author'].lower()
        ]
        return JsonResponse({'query': query, 'results': results})
```

**4. Edit `catalog/urls.py`:**

```python
from django.urls import path
from . import views

app_name = 'catalog'

urlpatterns = [
    path('', views.book_list, name='book_list'),
    path('<int:pk>/', views.book_detail, name='book_detail'),
    path('search/', views.BookSearchView.as_view(), name='book_search'),
]
```

**5. Test:**

```bash
python manage.py runserver

# Test with curl
curl http://127.0.0.1:8000/books/
curl http://127.0.0.1:8000/books/1/
curl http://127.0.0.1:8000/books/99/  # Should return 404
curl -X POST -d "query=django" http://127.0.0.1:8000/books/search/
```
</details>

<br>

### 🧠 EXERCISE 🧠, Add Request Information View

---

Create a debug view that displays request information:

1. Create a FBV `debug_request` that returns:
   - HTTP method
   - Path
   - Query parameters (if any)
   - User (authenticated or anonymous)
   - User-Agent header
2. Add URL pattern at `/debug/`
3. Test with: `http://127.0.0.1:8000/debug/?foo=bar&page=2`

<details>
    <summary>▶️ Solution</summary>
    
**1. Add to `catalog/views.py`:**

```python
def debug_request(request):
    """Display debug information about the request."""
    info = {
        'method': request.method,
        'path': request.path,
        'query_params': dict(request.GET),
        'user': str(request.user),
        'is_authenticated': request.user.is_authenticated,
        'user_agent': request.META.get('HTTP_USER_AGENT', 'Unknown'),
    }
    return JsonResponse(info, json_dumps_params={'indent': 2})
```

**2. Add to `catalog/urls.py`:**

```python
urlpatterns = [
    # ... existing patterns ...
    path('debug/', views.debug_request, name='debug'),
]
```

**3. Test:**

```bash
curl "http://127.0.0.1:8000/books/debug/?foo=bar&page=2"
```

Expected output:
```json
{
  "method": "GET",
  "path": "/books/debug/",
  "query_params": {"foo": ["bar"], "page": ["2"]},
  "user": "AnonymousUser",
  "is_authenticated": false,
  "user_agent": "curl/7.68.0"
}
```
</details>

---